In [0]:
# ============================================================
# FIGURE 1 — H2 TARGET DISTRIBUTION
# Self-contained: loads Notebook 02 output tables
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Load final H2 train/test datasets
# ------------------------------------------------------------

hydrogen_train = (
    spark.table("ml_perovskites.ml_hydrogen_train")
    .toPandas()
)

hydrogen_test = (
    spark.table("ml_perovskites.ml_hydrogen_test")
    .toPandas()
)

# ------------------------------------------------------------
# Identify target column
# ------------------------------------------------------------

TARGET_COL = "hydrogen_target"

if TARGET_COL not in hydrogen_train.columns:
    raise RuntimeError(
        f"'{TARGET_COL}' not found in training table."
    )

if TARGET_COL not in hydrogen_test.columns:
    raise RuntimeError(
        f"'{TARGET_COL}' not found in test table."
    )

# ------------------------------------------------------------
# Convert target to numeric
# ------------------------------------------------------------

train_h2 = pd.to_numeric(
    hydrogen_train[TARGET_COL],
    errors="coerce"
).dropna()

test_h2 = pd.to_numeric(
    hydrogen_test[TARGET_COL],
    errors="coerce"
).dropna()

# ------------------------------------------------------------
# Log1p transformation
# ------------------------------------------------------------

train_log = np.log1p(train_h2)
test_log = np.log1p(test_h2)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.hist(
    train_log,
    bins=30,
    alpha=0.65,
    label="Training set"
)

plt.hist(
    test_log,
    bins=30,
    alpha=0.65,
    label="Test set"
)

plt.xlabel(r"$\log(1 + H_2)$")
plt.ylabel("Number of observations")
plt.title("Distribution of hydrogen-production target")

plt.legend()

plt.tight_layout()

plt.savefig(
    "fig1_h2_target_distribution_log.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Audit
# ------------------------------------------------------------

print("=" * 70)
print("FIGURE 1 — H₂ TARGET DISTRIBUTION")
print("=" * 70)

print("Training observations:", len(train_h2))
print("Test observations:", len(test_h2))

print(
    "Training maximum H₂:",
    train_h2.max()
)

print(
    "Test maximum H₂:",
    test_h2.max()
)

print(
    "Training compounds:",
    hydrogen_train["Compound"].nunique()
)

print(
    "Test compounds:",
    hydrogen_test["Compound"].nunique()
)

print("\nFigure generated successfully.")

In [0]:
# ============================================================
# FIGURE — COMPOUND-LEVEL TARGET STRATA
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

def assign_h2_stratum(x):

    if x == 0:
        return "zero"
    elif x < 100:
        return "very_low"
    elif x < 1000:
        return "low"
    elif x < 10000:
        return "medium"
    elif x < 100000:
        return "high"
    else:
        return "extreme"


train_compound_summary = (
    hydrogen_train
    .groupby("Compound")["hydrogen_target"]
    .max()
    .apply(assign_h2_stratum)
    .value_counts()
)

test_compound_summary = (
    hydrogen_test
    .groupby("Compound")["hydrogen_target"]
    .max()
    .apply(assign_h2_stratum)
    .value_counts()
)

strata_order = [
    "zero",
    "very_low",
    "low",
    "medium",
    "high",
    "extreme"
]

plot_df = pd.DataFrame({
    "Train": train_compound_summary.reindex(
        strata_order,
        fill_value=0
    ),
    "Test": test_compound_summary.reindex(
        strata_order,
        fill_value=0
    )
})

ax = plot_df.plot(
    kind="bar",
    figsize=(9, 5)
)

ax.set_xlabel("H₂ target stratum")
ax.set_ylabel("Number of unique compounds")
ax.set_title(
    "Compound-level target stratification in train and test sets"
)

plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(
    "compound_level_target_stratification.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [0]:
# ============================================================
# FIGURE — MATERIAL VS MATERIAL + THERMO
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

comparison = pd.DataFrame({
    "Model": [
        "Kernel Ridge",
        "Kernel Ridge",
        "Random Forest",
        "Random Forest"
    ],
    "Feature group": [
        "Material only",
        "Material + thermo",
        "Material only",
        "Material + thermo"
    ],
    "CV_R2": [
        -0.1040411689,
        -0.1563612772,
        0.0222077915,
        0.0418347294
    ]
})

plot_df = comparison.pivot(
    index="Model",
    columns="Feature group",
    values="CV_R2"
)

ax = plot_df.plot(
    kind="bar",
    figsize=(8, 5)
)

ax.set_xlabel("Model")
ax.set_ylabel("Grouped cross-validation $R^2$")
ax.set_title(
    "Effect of thermodynamic descriptor enrichment"
)

ax.axhline(
    0,
    linewidth=1
)

plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(
    "thermodynamic_descriptor_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()